In [1]:
import Pkg; Pkg.add("CairoMakie")
import Pkg; Pkg.add("Statistics")
import Pkg; Pkg.add("FilePathsBase")
import Pkg; Pkg.add("RCall")

   Resolving package versions...
      Compat entries added for 
  No Changes to `C:\Specialet\Julia\Project.toml`
  No Changes to `C:\Specialet\Julia\Manifest.toml`
Precompiling project...
   6838.9 ms  ✓ gIVBMA
  1 dependency successfully precompiled in 28 seconds. 426 already precompiled.
   Resolving package versions...
      Compat entries added for 
  No Changes to `C:\Specialet\Julia\Project.toml`
  No Changes to `C:\Specialet\Julia\Manifest.toml`
   Resolving package versions...
      Compat entries added for 
  No Changes to `C:\Specialet\Julia\Project.toml`
  No Changes to `C:\Specialet\Julia\Manifest.toml`
   Resolving package versions...
      Compat entries added for 
  No Changes to `C:\Specialet\Julia\Project.toml`
  No Changes to `C:\Specialet\Julia\Manifest.toml`


In [2]:
using Pkg
Pkg.activate(@__DIR__)     # repo root
Pkg.instantiate()                           # one-time download

# QoL — hot-reload & progress bars
using Revise


  Activating project at `c:\Specialet\Julia`


In [3]:
# --- 0) CLEAN START -----------------------------------------------------------
# Restart your Julia kernel before running this cell.

using Random, CSV, DataFrames, LinearAlgebra, Statistics
using gIVBMA

# --- 1) PATHS ----------------------------------------------------------------
# Set the project root 

root = @__DIR__

datadir  = joinpath(root)
resdir   = joinpath(root, "results")
mkpath(resdir)

# --- 2) LOAD DATA -------------------------------------------------------------
df = CSV.read(joinpath(datadir, "acemoglu64.csv"), DataFrame)
# --- 3) BUILD MATRICES (one endogenous: avexpr) -------------------------------
yvar  = :logpgp95
xvars = [:avexpr]

# candidate Z = all numeric except y and X
num_syms = Symbol.(names(df, Number))
candZ    = setdiff(num_syms, vcat(yvar, xvars))

# remove true constants
candZ = filter(nm -> std(df[!, nm]) > 0, candZ)

excludeZ = [:imr95, :leb95]              # add more here if you like
candZ    = setdiff(candZ, excludeZ)

# build X, Z, y
Xfull = Matrix{Float64}(df[:, xvars])                  # n×1
Zfull = Matrix{Float64}(df[:, candZ])                  # n×p
y     = Vector{Float64}(df[:, yvar])


# sanity checks
@assert size(y,1) == size(Xfull,1) == size(Zfull,1)
@assert rank(Zfull)   == size(Zfull,2)     
@assert rank(Xfull) == size(Xfull,2)

# --- 4) RUN gIVBMA (one endogenous => dist has length 2) ---------------------
Random.seed!(1453)   # for reproducibility
iters = 10_000
burn  = 2_000

p = size(Zfull, 2)

res_bric = givbma(
    y, Xfull, Zfull;
    iter=iters, burn=burn,
    dist=["Gaussian","Gaussian"],
    g_prior="BRIC"
)

res_hg = givbma(
    y, Xfull, Zfull;
    iter=iters, burn=burn,
    dist=["Gaussian","Gaussian"],
    g_prior="hyper-g/n",
    m = [p/2, p/2]  # m[1] for outcome (W), m[2] for treatment (M)
)

# --- 5) SIMPLE OUTPUT ---------------------------------------------------------
# endogenous coefficient τ
τ_bric = (mean(vec(res_bric.τ)), quantile(vec(res_bric.τ), [.025,.975]))
τ_hg   = (mean(vec(res_hg.τ)),   quantile(vec(res_hg.τ),   [.025,.975]))

println("τ (BRIC): mean=$(round(τ_bric[1], digits=3))  95% CI=$(round.(τ_bric[2], digits=3))")
println("τ (HG/n): mean=$(round(τ_hg[1],   digits=3))  95% CI=$(round.(τ_hg[2],   digits=3))")

# PIPs for outcome model (L) and treatment model (M)
pip_L_bric = vec(mean(res_bric.L; dims=2))
pip_L_hg   = vec(mean(res_hg.L;   dims=2))
println("\nTop 10 L-PIPs (HG/n):")
perm = sortperm(pip_L_hg, rev=true)[1:min(10, length(pip_L_hg))]
for j in perm
    println(rpad(string(candZ[j]),20), "  ", round(pip_L_hg[j], digits=3))
end




τ (BRIC): mean=0.716  95% CI=[0.293, 1.052]
τ (HG/n): mean=0.826  95% CI=[0.614, 1.093]

Top 10 L-PIPs (HG/n):
catho80               0.375
humid3                0.028
edes1975              0.028
no_cpm80              0.025
goldm                 0.017
indtime               0.016
sjlofr                0.011
avelf                 0.011
asia                  0.01
meantemp              0.008


In [4]:
# Which fields are available in the result?
fieldnames(typeof(res_hg))


(:y, :X, :Z, :W, :dist, :α, :τ, :β, :Γ, :Δ, :Σ, :L, :M, :G, :Q, :r, :ν)

In [5]:
using Arrow

outdir = joinpath(@__DIR__, "results")
mkpath(outdir)

# --- hyper-g/n prior ---
tau = vec(res_hg.τ)                 # only one endogenous regressor
sigma12 = vec(res_hg.Σ[1, 2, :])    # covariance between y- and x-residuals

Arrow.write(joinpath(outdir, "acemoglu_posterior_hg.arrow"),
            (; tau, sigma12))
println("Posterior draws saved → results/acemoglu_posterior_hg.arrow")

# --- BRIC prior ---
tau = vec(res_bric.τ)
sigma12 = vec(res_bric.Σ[1, 2, :])

Arrow.write(joinpath(outdir, "acemoglu_posterior_bric.arrow"),
            (; tau, sigma12))
println("Posterior draws saved → results/acemoglu_posterior_bric.arrow")




Posterior draws saved → results/acemoglu_posterior_hg.arrow
Posterior draws saved → results/acemoglu_posterior_bric.arrow


In [6]:
using CairoMakie, Statistics



β_mean = τ_hg[1]
β_low, β_high = τ_hg[2]   # unpack [0.025, 0.975] quantiles

p = Figure(fontsize = 11)
ax = Axis(p[1,1]; xlabel = "β", ylabel = "Density")

# posterior density curve
lines!(ax, rbw(res_hg)[1],
       label = "gIVBMA hyper-g/n",
       color = Makie.wong_colors()[2])

# 95% interval lines (using your existing quantiles)
vlines!(ax, [β_low, β_high], color = :black, linestyle = :dash)

# (optional) add a line at the mean as well:
vlines!(ax, [β_mean], color = :black, linestyle = :dot)

axislegend(ax; position = :lt)

save(joinpath(outdir, "acemoglu_posteriors.png"), p; px_per_unit = 2)


In [7]:
#############################################################
# ▌Build LaTeX table (Acemoglu, one endogenous)    #
#############################################################
using Statistics, Printf
using DataFrames

# helper → (mean, 2.5 %, 97.5 %)
poststats(v) = begin
    v = vec(v)  # flatten any Matrix{Float64} to a Vector
    (mean(v), quantile(v, 0.025), quantile(v, 0.975))
end

# LaTeX escape for instruments (handles Symbols)
latex_escape(s::AbstractString) = replace(s, "_" => "\\_")
latex_escape(s) = latex_escape(string(s))

# --- coefficient summaries: hyper-g/n only -------------------------------
beta_hg = poststats(res_hg.τ)
Σ12_hg  = poststats(res_hg.Σ[1,2,:])

beta_hg_str = @sprintf("%.2f [%.2f, %.2f]", beta_hg[1], beta_hg[2], beta_hg[3])
Σ12_hg_str  = @sprintf("%.2f [%.2f, %.2f]", Σ12_hg[1], Σ12_hg[2], Σ12_hg[3])

# --- Top 10 L–PIPs -------------------------------------------------------
pip_hg = vec(mean(res_hg.L; dims = 2))

pips = DataFrame(
    variable = candZ,
    PIP_hg   = pip_hg
)

sort!(pips, :PIP_hg, rev = true)
pips_top = first(pips, 10)

pip_rows = join(
    ["$(latex_escape(pips_top.variable[i])) & $(round(pips_top.PIP_hg[i], digits=3))"
        for i in 1:nrow(pips_top)],
    " \\\\\n"
)




# --- Top 5 instrument inclusions (M and ψ|M=1) ----------------------------
pip_M = vec(mean(res_hg.M; dims=2))
Ψ = res_hg.Δ   

# Reshape ψ if needed
Ψmat = ndims(Ψ) == 2 ? Ψ : reshape(Ψ[:, 1, :], size(Ψ,1), size(Ψ,3))

psi_if_included = [
    let m = res_hg.M[i, :] .== 1
        any(m) ? mean(Ψmat[i, m]) : NaN
    end
    for i in 1:size(Ψmat, 1)
]

tbl_M = DataFrame(var = candZ, PIP_M = pip_M, psi_if_included = psi_if_included)
sort!(tbl_M, :PIP_M, rev=true)
tbl_M_top = first(tbl_M, 5)

instr_rows = join(
    ["$(latex_escape(tbl_M_top.var[i])) & $(round(tbl_M_top.PIP_M[i], digits=3)) & $(round(tbl_M_top.psi_if_included[i], digits=3))"
        for i in 1:nrow(tbl_M_top)],
    " \\\\\n"
)

# --- Build LaTeX table ----------------------------------------------------
latex = """
\\begin{table}[h]
\\centering
\\begin{tabular}{lc}
\\toprule
                & gIVBMA hyper-g/n \\\\
\\midrule
\$\\beta\$        & $beta_hg_str \\\\
\$\\sigma_{12}\$  & $Σ12_hg_str \\\\
\\midrule
\\multicolumn{2}{l}{\\textbf{Top 10 L-PIPs (Hyper-g/n)}} \\\\
\\multicolumn{2}{l}{
\\begin{tabular}{ll}
Variable & PIP \\\\
$pip_rows \\\\
\\end{tabular}
} \\\\
\\midrule
\\multicolumn{2}{l}{\\textbf{Top 5 instrument inclusions (M and \$\\psi\\mid M=1\$)}} \\\\
\\multicolumn{2}{l}{
\\begin{tabular}{lll}
Variable & PIP(M) & \$\\psi \\mid M=1\$ \\\\
$instr_rows \\\\
\\end{tabular}
} \\\\
\\bottomrule
\\end{tabular}
\\caption{Posterior summaries, top L-PIP values, and top instrument inclusion probabilities (ψ) for the Hyper-g/n prior.}
\\end{table}
"""

# --- Write to file --------------------------------------------------------
outdir = joinpath(@__DIR__, "results"); mkpath(outdir)
open(joinpath(outdir, "acemoglu_results.tex"), "w") do io
    print(io, latex)
end

println("LaTeX table written to results/acemoglu_results.tex")

N = size(res_hg.y, 1)   
println("N used in estimation = ", N)



LaTeX table written to results/acemoglu_results.tex
N used in estimation = 58


In [8]:
using Random, Statistics

"""
Run gIVBMA for several prior mean model sizes m_M
and print τ, σ₁₂, and average PIP over M for each run.

Arguments:
- y, X, Z : your usual givbma inputs
- mM_list : vector of prior mean treatment model sizes (for M)
- iter, burn, gprior, dist : passed to givbma

Returns:
- a vector of NamedTuples with the summaries for later use
"""
function run_givbma_mgrid(y, X, Z;
                          mM_list = Float64[],
                          iter::Int = 10_000,
                          burn::Int = 2_000,
                          gprior::String = "hyper-g/n",
                          dist::Vector{String} = repeat(["Gaussian"], size(X,2) + 1))

    p = size(Z, 2)
    isempty(mM_list) && (mM_list = [p/2, p/3, p/4, 5.0])  # sensible defaults

    results = NamedTuple[]

    for mM in mM_list
        println("\n==============================")
        println("Prior mean |M| = m_M = $(round(mM, digits=3))  (k = p = $p)")
        println("==============================")

        # outcome has no W, so m = [0.0, mM]
        m_vec = [mM, mM]

        # fix seed so changes reflect only the prior, not random noise
        Random.seed!(42)
        res = givbma(y, X, Z;
                     iter = iter,
                     burn = burn,
                     dist = dist,
                     g_prior = gprior,
                     m = m_vec)

        # τ: structural coefficient(s) on X in outcome equation
        τ_draws = vec(res.τ)
        τ_mean  = mean(τ_draws)
        τ_ci    = quantile(τ_draws, [0.025, 0.975])

        # σ₁₂: covariance between y and the endogenous regressor
        σ12_draws = res.Σ[1, 2, :]
        σ12_mean  = mean(σ12_draws)
        σ12_ci    = quantile(σ12_draws, [0.025, 0.975])

        # PIPs for treatment equation (M block) and average PIP
        pip_M = mean(res.M; dims = 2)[:]   # length = k+p; here k=0 → length p
        avg_pip_M = mean(pip_M)
        pip_L = mean(res.L; dims = 2)[:]
        avg_pip_L = mean(pip_L)

        println("τ   mean = $(round(τ_mean,  digits=3)),  95% CI = $(round.(τ_ci,   digits=3))")
        println("σ₁₂ mean = $(round(σ12_mean,digits=3)),  95% CI = $(round.(σ12_ci, digits=3))")
        println("Avg PIP(M) over instruments = $(round(avg_pip_M, digits=3))")
        println("Avg PIP(L) over instruments = $(round(avg_pip_L, digits=3))")

        push!(results, (
            mM          = mM,
            tau_mean    = τ_mean,
            tau_ci      = τ_ci,
            sigma12_mean = σ12_mean,
            sigma12_ci   = σ12_ci,
            avg_pip_M   = avg_pip_M,
        ))
    end

    return results
end


p = size(Zfull, 2)  

res_grid = run_givbma_mgrid(
    y, Xfull, Zfull;
    mM_list = [p, p/1.2, p/1.5, p/2, p/3, p/4, p/10, p/100],        # try several prior model sizes
    iter    = 10_000,
    burn    = 2_000,
    gprior  = "hyper-g/n",
    dist    = ["Gaussian", "Gaussian"],
)



Prior mean |M| = m_M = 41.0  (k = p = 41)
τ   mean = 0.175,  95% CI = [-0.151, 0.457]
σ₁₂ mean = 0.189,  95% CI = [-0.199, 0.647]
Avg PIP(M) over instruments = 0.463
Avg PIP(L) over instruments = 0.61

Prior mean |M| = m_M = 34.167  (k = p = 41)
τ   mean = 0.805,  95% CI = [0.535, 1.122]
σ₁₂ mean = -0.787,  95% CI = [-1.468, -0.302]
Avg PIP(M) over instruments = 0.058
Avg PIP(L) over instruments = 0.011

Prior mean |M| = m_M = 27.333  (k = p = 41)
τ   mean = 0.793,  95% CI = [0.543, 1.096]
σ₁₂ mean = -0.748,  95% CI = [-1.436, -0.289]
Avg PIP(M) over instruments = 0.058
Avg PIP(L) over instruments = 0.009

Prior mean |M| = m_M = 20.5  (k = p = 41)
τ   mean = 0.8,  95% CI = [0.417, 1.093]
σ₁₂ mean = -0.795,  95% CI = [-1.463, -0.182]
Avg PIP(M) over instruments = 0.055
Avg PIP(L) over instruments = 0.01

Prior mean |M| = m_M = 13.667  (k = p = 41)
τ   mean = 0.818,  95% CI = [0.525, 1.099]
σ₁₂ mean = -0.839,  95% CI = [-1.473, -0.366]
Avg PIP(M) over instruments = 0.058
Avg PIP(L) over

8-element Vector{NamedTuple}:
 (mM = 41.0, tau_mean = 0.17514452588420198, tau_ci = [-0.15081317579957815, 0.45683824125160305], sigma12_mean = 0.18903766670585803, sigma12_ci = [-0.19893245090601697, 0.6467392697959392], avg_pip_M = 0.4634146341463415)
 (mM = 34.16666666666667, tau_mean = 0.8047558960647091, tau_ci = [0.535060965075051, 1.121855431277185], sigma12_mean = -0.7865694577932223, sigma12_ci = [-1.468028824236179, -0.302047540196215], avg_pip_M = 0.058393292682926826)
 (mM = 27.333333333333332, tau_mean = 0.7928051327828773, tau_ci = [0.5434978107151099, 1.0959590160500847], sigma12_mean = -0.7478030034037327, sigma12_ci = [-1.4355136719629065, -0.2892958742273747], avg_pip_M = 0.057875)
 (mM = 20.5, tau_mean = 0.8004185890974569, tau_ci = [0.41695437912155264, 1.0932041022811878], sigma12_mean = -0.7948570667988119, sigma12_ci = [-1.4629023043929312, -0.18152494013937967], avg_pip_M = 0.05509451219512196)
 (mM = 13.666666666666666, tau_mean = 0.8182405930681198, tau_ci = [

In [9]:
function sweep_m_grid(y, X, Z;
                      scales_L = [1.0, 0.5, 1/3],
                      scales_M = [1.0, 0.5, 1/3],
                      iter::Int = 10_000,
                      burn::Int = 2_000,
                      gprior::String = "hyper-g/n",
                      dist::Vector{String} = repeat(["Gaussian"], size(X,2) + 1))

    p = size(Z, 2)
    results = NamedTuple[]

    for sL in scales_L, sM in scales_M
        mL = p * sL
        mM = p * sM
        m_vec = [mL, mM]

        println("\n======================================")
        println("m_L = $(round(mL,digits=3)) (p*$(round(sL,digits=3))),  m_M = $(round(mM,digits=3)) (p*$(round(sM,digits=3)))")
        println("======================================")

        Random.seed!(42)
        res = givbma(y, X, Z;
                     iter = iter,
                     burn = burn,
                     dist = dist,
                     g_prior = gprior,
                     m = m_vec)

        τ_draws   = vec(res.τ)
        τ_mean    = mean(τ_draws)
        τ_ci      = quantile(τ_draws, [0.025, 0.975])

        σ12_draws = res.Σ[1, 2, :]
        σ12_mean  = mean(σ12_draws)
        σ12_ci    = quantile(σ12_draws, [0.025, 0.975])

        pip_L     = mean(res.L; dims = 2)[:]
        pip_M     = mean(res.M; dims = 2)[:]
        avg_pip_L = isempty(pip_L) ? NaN : mean(pip_L)
        avg_pip_M = mean(pip_M)

        println("τ      mean = $(round(τ_mean,   digits=3)),  95% CI = $(round.(τ_ci,   digits=3))")
        println("σ₁₂    mean = $(round(σ12_mean, digits=3)),  95% CI = $(round.(σ12_ci, digits=3))")
        println("avg PIP(L)  = $(round(avg_pip_L,digits=3))")
        println("avg PIP(M)  = $(round(avg_pip_M,digits=3))")

        push!(results, (
            scale_L      = sL,
            scale_M      = sM,
            mL           = mL,
            mM           = mM,
            tau_mean     = τ_mean,
            tau_ci       = τ_ci,
            sigma12_mean = σ12_mean,
            sigma12_ci   = σ12_ci,
            avg_pip_L    = avg_pip_L,
            avg_pip_M    = avg_pip_M,
        ))
    end

    return results
end
sweep_results = sweep_m_grid(
    y, Xfull, Zfull;
    scales_L = [1.0, 0.5, 1/3],
    scales_M = [1.0, 0.5, 1/3],
    iter    = 10_000,
    burn    = 2_000,
    gprior  = "hyper-g/n",
    dist    = ["Gaussian", "Gaussian"],
)


m_L = 41.0 (p*1.0),  m_M = 41.0 (p*1.0)
τ      mean = 0.175,  95% CI = [-0.151, 0.457]
σ₁₂    mean = 0.189,  95% CI = [-0.199, 0.647]
avg PIP(L)  = 0.61
avg PIP(M)  = 0.463

m_L = 41.0 (p*1.0),  m_M = 20.5 (p*0.5)
τ      mean = 0.227,  95% CI = [-0.247, 0.706]
σ₁₂    mean = 0.103,  95% CI = [-0.6, 0.795]
avg PIP(L)  = 0.61
avg PIP(M)  = 0.047

m_L = 41.0 (p*1.0),  m_M = 13.667 (p*0.333)
τ      mean = 0.054,  95% CI = [-0.609, 0.735]
σ₁₂    mean = 0.419,  95% CI = [-0.68, 1.914]
avg PIP(L)  = 0.61
avg PIP(M)  = 0.045

m_L = 20.5 (p*0.5),  m_M = 41.0 (p*1.0)
τ      mean = 0.481,  95% CI = [0.147, 0.754]
σ₁₂    mean = -0.23,  95% CI = [-0.772, 0.291]
avg PIP(L)  = 0.04
avg PIP(M)  = 0.463

m_L = 20.5 (p*0.5),  m_M = 20.5 (p*0.5)
τ      mean = 0.8,  95% CI = [0.417, 1.093]
σ₁₂    mean = -0.795,  95% CI = [-1.463, -0.182]
avg PIP(L)  = 0.01
avg PIP(M)  = 0.055

m_L = 20.5 (p*0.5),  m_M = 13.667 (p*0.333)
τ      mean = 0.811,  95% CI = [0.587, 1.075]
σ₁₂    mean = -0.795,  95% CI = [-1.438,

9-element Vector{NamedTuple}:
 (scale_L = 1.0, scale_M = 1.0, mL = 41.0, mM = 41.0, tau_mean = 0.17514452588420198, tau_ci = [-0.15081317579957815, 0.45683824125160305], sigma12_mean = 0.18903766670585803, sigma12_ci = [-0.19893245090601697, 0.6467392697959392], avg_pip_L = 0.6097560975609756, avg_pip_M = 0.4634146341463415)
 (scale_L = 1.0, scale_M = 0.5, mL = 41.0, mM = 20.5, tau_mean = 0.22684092370339926, tau_ci = [-0.24690709636181601, 0.7056501435487009], sigma12_mean = 0.10260255103503646, sigma12_ci = [-0.6004676870366852, 0.7954775951593499], avg_pip_L = 0.6097560975609756, avg_pip_M = 0.04658231707317073)
 (scale_L = 1.0, scale_M = 0.3333333333333333, mL = 41.0, mM = 13.666666666666666, tau_mean = 0.05372355466897106, tau_ci = [-0.6088765512077459, 0.7353408481763418], sigma12_mean = 0.41884823677239497, sigma12_ci = [-0.6803083666801871, 1.913689034904901], avg_pip_L = 0.6097560975609756, avg_pip_M = 0.04471036585365854)
 (scale_L = 0.5, scale_M = 1.0, mL = 20.5, mM = 41.0, 

In [10]:
using DataFrames

function sweep_to_df_grid(sweep_results)
    rows = Dict[]
    for r in sweep_results
        push!(rows, Dict(
            :scale_L        => r.scale_L,
            :scale_M        => r.scale_M,
            :tau_mean       => r.tau_mean,
            :tau_ci_low     => r.tau_ci[1],
            :tau_ci_high    => r.tau_ci[2],
            :sigma12_mean   => r.sigma12_mean,
            :sigma12_ci_low => r.sigma12_ci[1],
            :sigma12_ci_high=> r.sigma12_ci[2],
            :avg_pip_L      => r.avg_pip_L,
            :avg_pip_M      => r.avg_pip_M,
        ))
    end
    return DataFrame(rows)
end

df_grid = sweep_to_df_grid(sweep_results)

using Printf

function make_latex_table_grid(df::DataFrame;
        caption = "Sensitivity of gIVBMA to prior mean model size",
        label   = "tab:givbma_prior_grid")

    buf = IOBuffer()

    println(buf, "\\begin{table}[h]")
    println(buf, "\\centering")
    println(buf, "\\begin{tabular}{ccccccc}")
    println(buf, "\\toprule")
    println(buf,
        "\$p_L\$ & \$p_M\$ & " *
        "\$\\hat{\\tau}\$ (95\\% CI) & " *
        "\$\\sigma_{12}\$ (95\\% CI) & " *
        "Avg PIP(L) & Avg PIP(M)\\\\"
    )
    println(buf, "\\midrule")

    for r in eachrow(df)
        @printf(buf,
            "p×%.2f & p×%.2f & %.3f [%.3f, %.3f] & %.3f [%.3f, %.3f] & %.3f & %.3f \\\\\n",
            r.scale_L, r.scale_M,
            r.tau_mean, r.tau_ci_low, r.tau_ci_high,
            r.sigma12_mean, r.sigma12_ci_low, r.sigma12_ci_high,
            r.avg_pip_L, r.avg_pip_M
        )
    end

    println(buf, "\\bottomrule")
    println(buf, "\\caption{$caption}")
    println(buf, "\\label{$label}")
    println(buf, "\\end{table}")

    return String(take!(buf))
end

latex_grid = make_latex_table_grid(df_grid)
println(latex_grid)



\begin{table}[h]
\centering
\begin{tabular}{ccccccc}
\toprule
$p_L$ & $p_M$ & $\hat{\tau}$ (95\% CI) & $\sigma_{12}$ (95\% CI) & Avg PIP(L) & Avg PIP(M)\\
\midrule
p×1.00 & p×1.00 & 0.175 [-0.151, 0.457] & 0.189 [-0.199, 0.647] & 0.610 & 0.463 \\
p×1.00 & p×0.50 & 0.227 [-0.247, 0.706] & 0.103 [-0.600, 0.795] & 0.610 & 0.047 \\
p×1.00 & p×0.33 & 0.054 [-0.609, 0.735] & 0.419 [-0.680, 1.914] & 0.610 & 0.045 \\
p×0.50 & p×1.00 & 0.481 [0.147, 0.754] & -0.230 [-0.772, 0.291] & 0.040 & 0.463 \\
p×0.50 & p×0.50 & 0.800 [0.417, 1.093] & -0.795 [-1.463, -0.182] & 0.010 & 0.055 \\
p×0.50 & p×0.33 & 0.811 [0.587, 1.075] & -0.795 [-1.438, -0.348] & 0.010 & 0.060 \\
p×0.33 & p×1.00 & 0.462 [0.126, 0.753] & -0.200 [-0.767, 0.304] & 0.041 & 0.463 \\
p×0.33 & p×0.50 & 0.760 [0.348, 1.079] & -0.720 [-1.399, -0.092] & 0.014 & 0.056 \\
p×0.33 & p×0.33 & 0.818 [0.525, 1.099] & -0.839 [-1.473, -0.366] & 0.011 & 0.058 \\
\bottomrule
\caption{Sensitivity of gIVBMA to prior mean model size}
\label{tab:givbm

In [11]:
# --- 0) CLEAN START -----------------------------------------------------------
# Restart your Julia kernel before running this cell.

using Random, CSV, DataFrames, LinearAlgebra, Statistics
using gIVBMA

# --- 1) PATHS ----------------------------------------------------------------
# Set the project root 

root = @__DIR__

datadir  = joinpath(root,)
resdir   = joinpath(root, "results")
mkpath(resdir)

# --- 2) LOAD DATA -------------------------------------------------------------
df = CSV.read(joinpath(datadir, "acemoglu64.csv"), DataFrame)
# --- 3) BUILD MATRICES (one endogenous: avexpr) -------------------------------
yvar  = :logpgp95
xvars = [:avexpr]

# candidate Z = all numeric except y and X
num_syms = Symbol.(names(df, Number))
candZ    = setdiff(num_syms, vcat(yvar, xvars))

# remove true constants
candZ = filter(nm -> std(df[!, nm]) > 0, candZ)

excludeZ = []             
candZ    = setdiff(candZ, excludeZ)

# build X, Z, y
Xfull = Matrix{Float64}(df[:, xvars])                  # n×1
Zfull = Matrix{Float64}(df[:, candZ])                  # n×p
y     = Vector{Float64}(df[:, yvar])


# sanity checks
@assert size(y,1) == size(Xfull,1) == size(Zfull,1)
@assert rank(Zfull)   == size(Zfull,2)     
@assert rank(Xfull) == size(Xfull,2)

# --- 4) RUN gIVBMA (one endogenous => dist has length 2) ---------------------
Random.seed!(42)   # for reproducibility
iters = 10_000
burn  = 2_000

res_bric = givbma(
    y, Xfull, Zfull;
    iter=iters, burn=burn,
    dist=["Gaussian","Gaussian"],
    g_prior="BRIC"
)

res_hg = givbma(
    y, Xfull, Zfull;
    iter=iters, burn=burn,
    dist=["Gaussian","Gaussian"],
    g_prior="hyper-g/n"
)

# --- 5) SIMPLE OUTPUT ---------------------------------------------------------
# endogenous coefficient τ
τ_bric = (mean(vec(res_bric.τ)), quantile(vec(res_bric.τ), [.025,.975]))
τ_hg   = (mean(vec(res_hg.τ)),   quantile(vec(res_hg.τ),   [.025,.975]))

println("τ (BRIC): mean=$(round(τ_bric[1], digits=3))  95% CI=$(round.(τ_bric[2], digits=3))")
println("τ (HG/n): mean=$(round(τ_hg[1],   digits=3))  95% CI=$(round.(τ_hg[2],   digits=3))")

# PIPs for outcome model (L) and treatment model (M)
pip_L_bric = vec(mean(res_bric.L; dims=2))
pip_L_hg   = vec(mean(res_hg.L;   dims=2))
println("\nTop 10 L-PIPs (HG/n):")
perm = sortperm(pip_L_hg, rev=true)[1:min(10, length(pip_L_hg))]
for j in perm
    println(rpad(string(candZ[j]),20), "  ", round(pip_L_hg[j], digits=3))
end




τ (BRIC): mean=0.844  95% CI=[0.612, 1.141]
τ (HG/n): mean=0.855  95% CI=[0.63, 1.144]

Top 10 L-PIPs (HG/n):
catho80               0.073
indtime               0.022
muslim80              0.019
meantemp              0.015
logem4                0.008
no_cpm80              0.007
temp5                 0.006
leb95                 0.003
sjlofr                0.003
africa                0.0


In [12]:
using Arrow

outdir = joinpath(@__DIR__, "results")
mkpath(outdir)

# --- hyper-g/n prior ---
tau = vec(res_hg.τ)                 # only one endogenous regressor
sigma12 = vec(res_hg.Σ[1, 2, :])    # covariance between y- and x-residuals

Arrow.write(joinpath(outdir, "acemoglu_posterior_hg.arrow"),
            (; tau, sigma12))
println("Posterior draws saved → results/acemoglu_posterior_hg.arrow")

# --- BRIC prior ---
tau = vec(res_bric.τ)
sigma12 = vec(res_bric.Σ[1, 2, :])

Arrow.write(joinpath(outdir, "acemoglu_posterior_bric.arrow"),
            (; tau, sigma12))
println("Posterior draws saved → results/acemoglu_posterior_bric.arrow")




Posterior draws saved → results/acemoglu_posterior_hg.arrow
Posterior draws saved → results/acemoglu_posterior_bric.arrow


In [13]:
#############################################################
# ▌ Build LaTeX table (Acemoglu, one endogenous)    #
#############################################################
using Statistics, Printf
using DataFrames

# helper → (mean, 2.5 %, 97.5 %)
poststats(v) = begin
    v = vec(v)  # flatten any Matrix{Float64} to a Vector
    (mean(v), quantile(v, 0.025), quantile(v, 0.975))
end

# LaTeX escape for instruments (handles Symbols)
latex_escape(s::AbstractString) = replace(s, "_" => "\\_")
latex_escape(s) = latex_escape(string(s))

# --- coefficient summaries: hyper-g/n only -------------------------------
beta_hg = poststats(res_hg.τ)
Σ12_hg  = poststats(res_hg.Σ[1,2,:])

beta_hg_str = @sprintf("%.2f [%.2f, %.2f]", beta_hg[1], beta_hg[2], beta_hg[3])
Σ12_hg_str  = @sprintf("%.2f [%.2f, %.2f]", Σ12_hg[1], Σ12_hg[2], Σ12_hg[3])

# --- Top 10 L–PIPs -------------------------------------------------------
pip_hg = vec(mean(res_hg.L; dims = 2))

pips = DataFrame(
    variable = candZ,
    PIP_hg   = pip_hg
)

sort!(pips, :PIP_hg, rev = true)
pips_top = first(pips, 10)

pip_rows = join(
    ["$(latex_escape(pips_top.variable[i])) & $(round(pips_top.PIP_hg[i], digits=3))"
        for i in 1:nrow(pips_top)],
    " \\\\\n"
)




# --- Top 5 instrument inclusions (M and ψ|M=1) ----------------------------
pip_M = vec(mean(res_hg.M; dims=2))
Ψ = res_hg.Δ   

# Reshape ψ if needed
Ψmat = ndims(Ψ) == 2 ? Ψ : reshape(Ψ[:, 1, :], size(Ψ,1), size(Ψ,3))

psi_if_included = [
    let m = res_hg.M[i, :] .== 1
        any(m) ? mean(Ψmat[i, m]) : NaN
    end
    for i in 1:size(Ψmat, 1)
]

tbl_M = DataFrame(var = candZ, PIP_M = pip_M, psi_if_included = psi_if_included)
sort!(tbl_M, :PIP_M, rev=true)
tbl_M_top = first(tbl_M, 5)

instr_rows = join(
    ["$(latex_escape(tbl_M_top.var[i])) & $(round(tbl_M_top.PIP_M[i], digits=3)) & $(round(tbl_M_top.psi_if_included[i], digits=3))"
        for i in 1:nrow(tbl_M_top)],
    " \\\\\n"
)

# --- Build LaTeX table ----------------------------------------------------
latex = """
\\begin{table}[h]
\\centering
\\begin{tabular}{lc}
\\toprule
                & gIVBMA hyper-g/n \\\\
\\midrule
\$\\beta\$        & $beta_hg_str \\\\
\$\\sigma_{12}\$  & $Σ12_hg_str \\\\
\\midrule
\\multicolumn{2}{l}{\\textbf{Top 10 L-PIPs (Hyper-g/n)}} \\\\
\\multicolumn{2}{l}{
\\begin{tabular}{ll}
Variable & PIP \\\\
$pip_rows \\\\
\\end{tabular}
} \\\\
\\midrule
\\multicolumn{2}{l}{\\textbf{Top 5 instrument inclusions (M and \$\\psi\\mid M=1\$)}} \\\\
\\multicolumn{2}{l}{
\\begin{tabular}{lll}
Variable & PIP(M) & \$\\psi \\mid M=1\$ \\\\
$instr_rows \\\\
\\end{tabular}
} \\\\
\\bottomrule
\\end{tabular}
\\caption{Posterior summaries, top L-PIP values, and top instrument inclusion probabilities (ψ) for the Hyper-g/n prior.}
\\end{table}
"""

# --- Write to file --------------------------------------------------------
outdir = joinpath(@__DIR__, "results"); mkpath(outdir)
open(joinpath(outdir, "acemoglu_results2.tex"), "w") do io
    print(io, latex)
end

println("LaTeX table written to results/acemoglu_results2.tex")

N = size(res_hg.y, 1)  
println("N used in estimation = ", N)



LaTeX table written to results/acemoglu_results2.tex
N used in estimation = 58


In [14]:
using gIVBMA

# To see the source of givbma:
@less givbma(y, Xfull, Zfull; iter=10, burn=2, g_prior="hyper-g/n")



function givbma(
    y::AbstractVector{<:Real},
    X::AbstractVecOrMat{<:Real},
    Z::AbstractMatrix{<:Real};
    iter::Integer = 2000,
    burn::Integer = 1000,
    dist::Vector{String} = repeat(["Gaussian"], size(X, 2) + 1),
    two_comp = false,
    ν::Union{Nothing, Number} = nothing,
    m::Union{AbstractVector, Nothing} = nothing,
    g_prior::String = "BRIC",
    r_prior::Distribution = Exponential(1)
)
    # if X is a vector turn it into an nx1 matrix
    if ndims(X) == 1
        X = permutedims(X)'
    end

    # Use default prior mean model size if not specified
    n = length(y)
    p = size(Z, 2)
    if isnothing(m)
        m = [p/2, p/2]
    end

    res = givbma_mcmc(y, X, Matrix{Float64}(undef, n, 0), Z, dist, two_comp, iter, burn, ν, m, g_prior, r_prior)

    return res
end


end


In [15]:
###############################################################################
# ▌ LOOCV LPS with global de-aliasing + fold-constant drop only     ##
#           Methods: gIVBMA (hyper-g/n) and TSLS                            ##
###############################################################################
using ProgressMeter, Statistics, LinearAlgebra, Printf
include("../Julia/Simulations/competing_methods.jl")

# 1 ▸ One-time global de-aliasing on full Z (remove only exact redundancies/constants)
function global_keep_cols(Zfull::AbstractMatrix)
    # drop true constants first
    keep0 = [std(view(Zfull, :, j)) > 0 for j in 1:size(Zfull, 2)]
    F = Matrix{Float64}(Zfull[:, keep0])
    if isempty(F)
        return Int[]
    end
    Q = qr(F, Val(true))
    r = rank(F)
    return findall(keep0)[Q.p[1:r]]  # indices into original Z
end

global_keep = global_keep_cols(Zfull)

# 2 ▸ Per-fold: use global_keep and only drop columns that are constant in the training split
fold_Z = function(Ztr::AbstractMatrix, Zte::AbstractMatrix, keep::Vector{Int})
    cols = [j for j in keep if std(view(Ztr, :, j)) > 0]  # drop fold-constant only
    return Matrix{Float64}(Ztr[:, cols]), Matrix{Float64}(Zte[:, cols])
end

# 3 ▸ LOOCV (1 endogenous ⇒ dist has length 2 here, as in your current setup)
function loocv(y::Vector, Xfull::AbstractMatrix, Zfull::AbstractMatrix;
               iters::Int = 10_000, burn::Int = 2_000)
    n = length(y)
    # 1: gIVBMA hyper-g/n   2: TSLS
    scores = zeros(n, 2)

    @showprogress desc = "LOOCV" for i in 1:n
        idx_tr, idx_te = setdiff(1:n, i), i:i
        ytr, Xtr, Ztr = y[idx_tr], Xfull[idx_tr, :], Zfull[idx_tr, :]
        yte, Xte, Zte = y[idx_te], Xfull[idx_te, :], Zfull[idx_te, :]

        Ztr_fr, Zte_fr = fold_Z(Ztr, Zte, global_keep)

        # gIVBMA  (hyper-g/n)
        scores[i, 1] = lps(
            givbma(ytr, Xtr, Ztr_fr;
                   iter = iters,
                   burn = burn,
                   dist = ["Gaussian", "Gaussian"],
                   g_prior = "hyper-g/n"),
            yte, Xte, Zte_fr
        )

        # TSLS
        scores[i, 2] = tsls(ytr, Xtr, Ztr_fr, yte, Xte, Zte_fr).lps
    end
    return scores
end

# 4 ▸ Run and summarise
scores  = loocv(y, Xfull, Zfull)
methods = ["gIVBMA hyper-g/n", "TSLS"]
meanLPS = round.(mean(scores; dims = 1)[:], digits = 3)

println("\nMean leave-one-out log-predictive score (lower = better):")
for (m, v) in zip(methods, meanLPS)
    @printf("  %-17s  %8.3f\n", m, v)
end


LOOCV 100%|██████████████████████████████████████████████| Time: 0:02:03



Mean leave-one-out log-predictive score (lower = better):
  gIVBMA hyper-g/n      0.749
  TSLS                  1.041
